# 1. Get set up

By the end of this notebook you will have pulled real economic data from the
US government and looked at it. That is genuinely all. It should take about
fifteen minutes, and most of that is waiting for two emails.

**You need to know:** how to run a notebook cell. That is it. If you have never
written pandas before, you are in the right place.

**A note on the shape of this.** Almost every data project starts the same
boring way: get access, get the data, look at it before you do anything clever.
People skip the looking and it costs them later. We are not going to skip it.

## Install

Three libraries. `fred-loader` and `census-loader` fetch public data and give
it readable names; `otter` does the statistics on it later.

If you are on your own machine, run this once. If a line fails, that is
usually a Python version older than 3.10.

In [ ]:
%pip install --quiet otter fred-loader census-loader matplotlib

## Get two free keys

Both agencies want to know who is calling. Neither charges, neither asks for a
card, and both issue the key immediately.

1. **FRED** (the St Louis Federal Reserve, which mirrors most US economic
   series): <https://fredaccount.stlouisfed.org/apikeys>
2. **Census** (population, income, housing, everything about people):
   <https://api.census.gov/data/key_signup.html>

### Where to put them

Make a file called `.env` next to this notebook, containing:

```
FRED_API_KEY=your_fred_key_here
CENSUS_API_KEY=your_census_key_here
```

**Why a file and not just typing the key into a cell.** A key pasted into a
notebook gets committed to git, and once a secret is in git history it is
effectively public forever. This habit costs you nothing now and saves you a
genuinely bad afternoon later. Add `.env` to your `.gitignore`.

In [ ]:
import os
from pathlib import Path

# Read the .env file by hand so you can see there is no magic in it.
env = Path(".env")
if env.exists():
    for line in env.read_text().splitlines():
        if "=" in line and not line.startswith("#"):
            key, _, value = line.partition("=")
            os.environ[key.strip()] = value.strip()

# Check, and say WHICH one is missing rather than failing later and vaguely.
for name in ("FRED_API_KEY", "CENSUS_API_KEY"):
    print(f"{name}: {'found' if os.getenv(name) else 'MISSING'}")

If either says MISSING, fix that before going on. A notebook that half works is
harder to debug than one that refuses to start.

## Your first series

FRED identifies everything by a code. Unemployment is `UNRATE`, inflation is
`CPIAUCSL`, and there are about 800,000 more. Nobody memorises these, which is
the entire reason `fred-loader` exists: you search in words and it finds the
code.

Start by looking for something rather than knowing it.

In [ ]:
import fred_loader as fl

# What does this library actually offer? Ask it, do not guess.
print([name for name in dir(fl) if not name.startswith("_")])

**That cell is a habit worth keeping.** `dir()` on any library tells you what it
exposes. You will use it more than you use documentation.

Now pull the unemployment rate. It is monthly, goes back to 1948, and everyone
has an intuition about it, which makes it a good first thing to look at: you can
tell immediately whether the numbers are wrong.

In [ ]:
unemployment = fl.pull_fred(["UNRATE"], start="1990-01-01")
unemployment.tail(10)

## Look at it before you do anything to it

Three questions, every single time you load data. They take one minute and they
catch most of the mistakes that would otherwise show up in your conclusion.

In [ ]:
# 1. How much is there, and what period does it cover?
print(f"rows: {len(unemployment)}")
print(f"from: {unemployment.index.min()}  to: {unemployment.index.max()}")

# 2. Is anything missing? A gap you do not know about becomes a wrong average.
print(f"missing values:\n{unemployment.isna().sum()}")

# 3. Are the numbers plausible? Unemployment is a percent, so single or low
#    double digits. If you see 0.04 the units are a fraction, not a percent,
#    and every number you produce afterwards is off by a hundred.
unemployment.describe()

## Now look at it properly

A table of numbers hides shape. A chart shows it instantly, and shape is what
you are actually after.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
unemployment.plot(ax=ax, legend=False)
ax.set_title("US unemployment rate, 1990 to now")
ax.set_ylabel("percent")
ax.set_xlabel("")
plt.show()

**Spend a moment on that chart.** You can see the early 90s recession, the
2008 crisis, and the near-vertical spike in 2020. If you could not, something
would be wrong with the data and you would want to know now rather than after
building an analysis on it.

This is the whole point of looking first. You already know roughly what US
unemployment did; use that knowledge as a test of the data, because later you
will load series you have no intuition about and the habit is what protects you.

## Two series, and the first real trap

One series is rarely interesting. Two lets you ask whether they move together.

But series have different frequencies. Unemployment is monthly; GDP is
quarterly. Put them in one table and three of every four GDP rows are blank.

In [ ]:
both = fl.pull_fred(["UNRATE", "GDP"], start="1990-01-01")
both.head(8)

See the gaps. Nothing is broken: quarterly data genuinely has no February
value, and the library is telling you the truth rather than inventing one.

**The thing people get wrong here** is filling those gaps without deciding what
the fill means. Carrying the last value forward says "GDP was unchanged until
the next reading", which is a claim about the world, not a formatting choice.
Averaging the monthly data down to quarters says something different again.

Either can be right. Choosing without noticing you chose is what is wrong.
Notebook 3 does this properly.

For now, drop the incomplete rows, which is the honest option when you have
plenty of data.

In [ ]:
quarterly = both.dropna()
print(f"{len(both)} rows became {len(quarterly)} once incomplete ones were dropped")
quarterly.head()

## What you now know

- Where to get real economic data, and that it is free
- Why a key lives in a file rather than in your code
- To check size, gaps and plausibility before anything else
- That combining two series raises a question about frequency, and that the
  question has no automatic answer

**Next:** [`02-your-first-question.ipynb`](02-your-first-question.ipynb) turns a
vague thought into something data can actually answer, and runs the regression.